<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-02-27

| Package | Version |
|---------|---------|
| **nnsight** | **0.6.1** |
| Python | 3.12.11 |
| torch | 2.10.0 |
| transformers | 5.2.0 |

</details>


# Accessing Intermediate Operations

`.output` and `.input` let you hook into a module's inputs and outputs. But what about the operations *inside* a module's forward pass? `.source` lets you access any intermediate operation — function calls, method calls, tensor operations — within a module's forward method.

## Setup

In [1]:
from nnsight import LanguageModel

model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

## Discovering Operations

Print `.source` on any module to see its forward method with all hookable operations labeled.

In [2]:
print(model.transformer.h[0].mlp.source)

                    * def forward(self, hidden_states: tuple[torch.FloatTensor] | None) -> torch.FloatTensor:
 self_c_fc_0    ->  0     hidden_states = self.c_fc(hidden_states)
 self_act_0     ->  1     hidden_states = self.act(hidden_states)
 self_c_proj_0  ->  2     hidden_states = self.c_proj(hidden_states)
 self_dropout_0 ->  3     hidden_states = self.dropout(hidden_states)
                    4     return hidden_states
                    5 


Each labeled line (like `self_c_fc_0`, `self_act_0`) is an operation you can access inside a trace. The number suffix distinguishes multiple calls to the same function.

Larger modules have more operations. Here's the attention module:

In [3]:
print(model.transformer.h[0].attn.source)

                                             * def forward(
                                             0     self,
                                             1     hidden_states: tuple[torch.FloatTensor] | None,
                                             2     past_key_values: Cache | None = None,
                                             3     cache_position: torch.LongTensor | None = None,
                                             4     attention_mask: torch.FloatTensor | None = None,
                                             5     encoder_hidden_states: torch.Tensor | None = None,
                                             6     encoder_attention_mask: torch.FloatTensor | None = None,
                                             7     output_attentions: bool | None = False,
                                             8     **kwargs,
                                             9 ) -> tuple[torch.Tensor | tuple[torch.Tensor], ...]:
                                  

## Getting an Intermediate Value

Access any labeled operation's `.output` inside a trace — just like you would with a module.

In [4]:
with model.trace("The Eiffel Tower is in the city of"):
    # Get the hidden states after the GELU activation (before projection)
    post_gelu = model.transformer.h[0].mlp.source.self_act_0.output.save()

print(f"Post-GELU shape: {post_gelu.shape}")

Post-GELU shape: torch.Size([1, 10, 3072])


<details class="admonition note">
<summary>How source works</summary>

When you access `.source`, nnsight rewrites the module's forward method to inject hooks into every operation. Each operation gets `.input`, `.inputs`, and `.output` properties — just like a regular module.

</details>

## Setting an Intermediate Value

You can also modify intermediate values. Here we zero out the MLP's post-GELU activations at layer 11 to see how it affects the prediction:

In [5]:
with model.trace("The Eiffel Tower is in the city of"):
    normal_logits = model.lm_head.output.save()

with model.trace("The Eiffel Tower is in the city of"):
    # Zero the MLP's GELU output at layer 11
    model.transformer.h[11].mlp.source.self_act_0.output[:] = 0
    modified_logits = model.lm_head.output.save()

print(f"Normal:      {model.tokenizer.decode(normal_logits[0, -1].argmax(dim=-1))}")
print(f"Zeroed GELU: {model.tokenizer.decode(modified_logits[0, -1].argmax(dim=-1))}")

Normal:       Paris
Zeroed GELU:  London


## Patching Between Layers

Transfer an intermediate value from one layer to another:

In [6]:
with model.trace("The Eiffel Tower is in the city of"):
    # Capture layer 0's post-GELU activations
    gelu_0 = model.transformer.h[0].mlp.source.self_act_0.output

    # Patch them into layer 5
    model.transformer.h[5].mlp.source.self_act_0.output = gelu_0

    logits = model.lm_head.output.save()

print(f"Patched MLP prediction: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")

Patched MLP prediction:  London


## Recursive Source Tracing

`.source` works recursively. If a labeled operation calls another function, you can chain `.source` to trace into it. For example, the attention interface function internally calls `scaled_dot_product_attention`:

In [7]:
with model.trace("The Eiffel Tower is in the city of"):
    # Trace into the attention interface → access the SDPA call's input (the query tensor)
    sdpa = model.transformer.h[0].attn.source.attention_interface_0.source
    query = sdpa.torch_nn_functional_scaled_dot_product_attention_0.input.save()

print(f"Query shape: {query.shape}")  # [batch, heads, seq_len, head_dim]

Query shape: torch.Size([1, 12, 10, 64])


<details class="admonition tip">
<summary>Viewing a specific operation</summary>

Print a specific operation to see it highlighted in its surrounding context:

```python
print(model.transformer.h[0].attn.source.self_c_proj_0)
```

This shows the operation with an arrow (`-->`) and surrounding lines for context.

</details>

<details class="admonition warning">
<summary>Don't chain .source through another .source on submodules</summary>

If a `.source` listing shows a submodule call (like `self.c_proj`), access it directly on the model — not through `.source`:

```python
# Wrong — don't chain through .source
model.transformer.h[0].attn.source.self_c_proj.source

# Correct — access the submodule directly
model.transformer.h[0].attn.c_proj.source
```

</details>